In [1]:
%pip install -q "transformers==4.57.1" "accelerate==1.11.0"
%pip install -q --no-deps "bitsandbytes==0.48.2"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import gc
import json
import re
import time
import unicodedata
from decimal import Decimal
from pathlib import Path

import torch
import pandas as pd
import transformers
import bitsandbytes as bnb

from PIL import Image
from IPython.display import display
from tqdm.auto import tqdm
from transformers import AutoProcessor, GenerationConfig, BitsAndBytesConfig, Qwen3VLForConditionalGeneration

In [4]:
ROOT = Path("/content/drive/MyDrive/MLLM/mordern_llm")
DOC_DIR = ROOT / "data/sroie"
DAY3_OUT = ROOT / "outputs/quantization"
DAY3_OUT.mkdir(parents=True, exist_ok=True)

manifest = json.loads((DOC_DIR / "manifest.json").read_text(encoding="utf-8"))

task_config = json.loads((DOC_DIR / "task_config.json").read_text(encoding="utf-8"))

FIELDS = tuple(task_config["fields"])
DOC_PROMPT = task_config["prompt"]
IMAGE_KWARGS = task_config["image_kwargs"]

lookup = {row["id"]: row for row in manifest["val"]}
probe_records = [lookup[doc_id] for doc_id in manifest["val_probe_ids"]]

ENV = {
    "gpu": torch.cuda.get_device_name(0),
    "torch": str(torch.__version__),
    "cuda": torch.version.cuda,
    "transformers": transformers.__version__,
    "bitsandbytes": bnb.__version__
}

print(ENV)
print("Validation receipts:", len(probe_records))

{'gpu': 'Tesla T4', 'torch': '2.14.1+cu130', 'cuda': '13.0', 'transformers': '4.57.1', 'bitsandbytes': '0.48.2'}
Validation receipts: 20


**Checkpoints and NF4 recipe**

In [ ]:
CHECKPOINTS = {
    "2b": {
        "model_id": "Qwen/Qwen3-VL-2B-Instruct",
        "revision": "89644892e4d85e24eaac8bacfd4f463576704203"
    },
    "4b": {
        "model_id": "Qwen/Qwen3-VL-4B-Instruct",
        "revision": "ebb281ec70b05090aa6165b016eac8ec08e71b17"
    }
}

SKIP_MODULES = ["visual", "lm_head"]
def nf4_config():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
        llm_int8_skip_modules=SKIP_MODULES
    )

**prediction of one receipt**

In [ ]:
def user_messages():
    return [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": DOC_PROMPT}
        ]
    }]

@torch.inference_mode()
def predict_one(model, processor, generation_config, image_path):
    with Image.open(image_path) as source:
        image = source.convert("RGB")
    
    prompt = processor.apply_chat_template(
        user_messages(),
        tokenize=False,
        add_generation_prompt=True
    )
    batch = processor(
        text=[prompt],
        images=[image],
        images_kwargs=IMAGE_KWARGS,
        add_special_tokens=False,
        return_tensors="pt"
    ).to(model.device)
    
    input_length = batch["input_ids"].shape[1]
    merge = model.config.vision_config.spatial_merge_size
    visual_tokens = int(batch["image_grid_thw"].prod(dim=-1).sum().item()) // merge**2
    
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    
    output = model.generate(
        **batch,
        generation_config=generation_config,
        use_model_defaults=False,
        do_sample=False
    )
    
    torch.cuda.synchronize()
    seconds = time.perf_counter() - start
    peak = torch.cuda.max_memory_allocated() / 1024**3
    new_ids = output[:, input_length:]

    return {
        "raw_answer": processor.batch_decode(
            new_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0],
        "input_tokens": input_length,
        "visual_tokens": visual_tokens,
        "new_tokens": new_ids.shape[1],
        "latency_s": seconds,
        "peak_gpu_GiB": peak
    }

In [7]:
def normal(field, value):
    if not isinstance(value, str) or not value.strip():
        return None

    text = " ".join(
        unicodedata.normalize("NFKC", value).casefold().split()
    )
    if field != "total":
        return text
    
    text = re.sub(r"^(?:rm|\$)\s*", "", text)
    if not re.fullmatch(
        r"[+-]?(?:\d+|\d{1,3}(?:,\d{3})+)(?:\.\d+)?",
        text
    ):
        return None

    amount = Decimal(text.replace(",", ""))
    return format(amount.normalize(), "f")

In [8]:
def score_receipt(raw, reference):
    try:
        json.loads(raw)
        raw_valid = True
    except ValueError:
        raw_valid = False
    
    text = raw.strip()
    lines = text.splitlines()
    wrapped = (
        len(lines) >= 3
        and lines[0].strip().lower() in ("```json", "```")
        and lines[-1].strip() == "```"
    )
    
    if wrapped:
        text = "\n".join(lines[1:-1])
    
    try:
        prediction = json.loads(text)
    except ValueError:
        prediction = None

    schema = (
        isinstance(prediction, dict)
        and set(prediction) == set(FIELDS)
        and all(
            isinstance(value, str) or value is None
            for value in prediction.values()
        )
    )
    correct = {}
    for field in FIELDS:
        if not schema:
            correct[field] = False
        elif prediction[field] is None or reference[field] is None:
            correct[field] = prediction[field] is reference[field]
        else:
            pred = normal(field, prediction[field])
            gold = normal(field, reference[field])
            correct[field] = pred is not None and pred == gold

    return {
        "prediction": prediction,
        "raw_json_valid": raw_valid,
        "schema_valid": schema,
        "used_fence_cleanup": wrapped,
        **{field + "_correct": value for field, value in correct.items()},
        "all_correct": all(correct.values())
    }

In [9]:
def run_setting(name, size, quantized):
    checkpoint = CHECKPOINTS[size]
    processor = AutoProcessor.from_pretrained(
        checkpoint["model_id"],
        revision=checkpoint["revision"],
        use_fast=False
    )
    
    extra = {}
    if quantized:
        extra["quantization_config"] = nf4_config()
    
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        checkpoint["model_id"],
        revision=checkpoint["revision"],
        dtype=torch.float16,
        device_map={"": 0},
        attn_implementation="sdpa",
        low_cpu_mem_usage=True,
        **extra
    ).eval()
    
    quantized_names = [
        name for name, module in model.named_modules()
        if isinstance(module, bnb.nn.Linear4bit)
    ]
    
    print(name, "| 4-bit Linear layers:", len(quantized_names))
    if quantized_names:
        print("First converted layer:", quantized_names[0])
        
    generation_config = GenerationConfig(
        max_new_tokens=task_config["max_new_tokens"],
        do_sample=False,
        use_cache=True,
        bos_token_id=model.generation_config.bos_token_id,
        eos_token_id=model.generation_config.eos_token_id,
        pad_token_id=processor.tokenizer.pad_token_id
    )
    
    run = {
        "source": "fresh",
        "run_info": {
            "setting": name,
            **checkpoint,
            "quantized": quantized,
            "compute_dtype": "torch.float16",
            "skip_modules": SKIP_MODULES if quantized else [],
            "environment": ENV,
            "task_config": task_config
        },
        "rows": []
    }

    _ = predict_one(model, processor, generation_config, probe_records[0]["image_path"])
    
    for record in tqdm(probe_records, desc=name):
        result = predict_one(model, processor, generation_config, record["image_path"])

        run["rows"].append({
            "id": record["id"],
            "reference": record["reference"],
            **result,
            **score_receipt(result["raw_answer"], record["reference"])
        })

        (DAY3_OUT / f"{name}.json").write_text(json.dumps(run, ensure_ascii=False, indent=2), encoding="utf-8")

    del model, processor
    return run

**Baseline compatibility and execution**

In [10]:
FORCE_FP16_RERUN = True

baseline_path =  ROOT / "outputs/documents/day2_base_validation20.json"

old_baseline = (
    json.loads(baseline_path.read_text(encoding="utf-8"))
    if baseline_path.exists()
    else None
)

def compatible_baseline(saved):
    if saved is None:
        return False
    
    info = saved["run_info"]
    rows = saved["rows"]
    return (
        info.get("model") == CHECKPOINTS["2b"]["model_id"]
        and info.get("dtype") == "torch.float16"
        and info.get("gpu") == ENV["gpu"]
        and info.get("transformers") == ENV["transformers"]
        and info.get("batch_size") == 1
        and info.get("task_config") == task_config
        and [row["id"] for row in rows] == manifest["val_probe_ids"]
        and all(row["reference"] == lookup[row["id"]]["reference"] for row in rows)
    )

if "model" in globals():
    del model
    
gc.collect()
torch.cuda.empty_cache()
runs = {}

if (not FORCE_FP16_RERUN and compatible_baseline(old_baseline)):
    runs["2b_fp16"] = {
        **old_baseline,
        "source": "day2_reused"
    }
    print("Reusing Day 2 FP16 baseline.")
else:
    runs["2b_fp16"] = run_setting("2b_fp16", "2b", False)

gc.collect()
torch.cuda.empty_cache()

for name, size in (("2b_nf4", "2b"), ("4b_nf4", "4b")):
    runs[name] = run_setting(name, size, True)
    gc.collect()
    torch.cuda.empty_cache()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (23 outputs left out)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

4b_nf4 | 4-bit Linear layers: 252
First converted layer: model.language_model.layers.0.self_attn.q_proj


4b_nf4:   0%|          | 0/20 [00:00<?, ?it/s]

**Comparison table and configuration save**

In [11]:
score_columns = [
    "raw_json_valid",
    "schema_valid",
    "company_correct",
    "date_correct",
    "total_correct",
    "all_correct"
]

summary_rows = []
for setting, run in runs.items():
    frame = pd.DataFrame(run["rows"])
    assert len(frame) == len(probe_records)

    summary_rows.append({
        "setting": setting,
        "source": run["source"],
        "receipts": len(frame),
        **{
            column + "_pct": 100 * frame[column].mean()
            for column in score_columns
        },
        "mean_latency_s": frame["latency_s"].mean(),
        "peak_gpu_GiB": frame["peak_gpu_GiB"].max(),
        "mean_new_tokens": frame["new_tokens"].mean()
    })

comparison = pd.DataFrame(summary_rows)
display(comparison.round(3))

comparison.to_csv(DAY3_OUT / "day3_comparison.csv", index=False)
(DAY3_OUT / "all_runs.json").write_text(json.dumps(runs, ensure_ascii=False, indent=2), encoding="utf-8")

,setting,source,receipts,raw_json_valid_pct,schema_valid_pct,company_correct_pct,date_correct_pct,total_correct_pct,all_correct_pct,mean_latency_s,peak_gpu_GiB,mean_new_tokens
0,2b_fp16,fresh,20,90.0,100.0,90.0,85.0,90.0,70.0,2.468,4.175,47.45
1,2b_nf4,fresh,20,0.0,100.0,85.0,85.0,85.0,65.0,4.477,2.240,50.95
2,4b_nf4,fresh,20,100.0,100.0,95.0,100.0,95.0,90.0,5.431,3.566,45.05


55659

In [12]:
qlora_base_config = {
    **CHECKPOINTS["2b"],
    "dtype": "float16",
    "quantization_config": nf4_config().to_dict(),
    "attn_implementation": "sdpa",
    "micro_batch_size": 1,
    "max_sequence": task_config["max_sequence"],
    "manifest_path": str(DOC_DIR / "manifest.json"),
    "task_config_path": str(DOC_DIR / "task_config.json")
}

(DAY3_OUT / "qlora_base_config.json").write_text(json.dumps(qlora_base_config, ensure_ascii=False, indent=2), encoding="utf-8")

902

### QLoRA Fine-tuning

$$y = W_{\text{frozen}}x + \frac{\alpha}{r}B(Ax)$$


In [13]:
%pip install -q "transformers==4.57.1" "accelerate==1.11.0"
%pip install -q --no-deps "bitsandbytes==0.48.2" "peft==0.17.1"

In [14]:
import gc
import json
import re
import unicodedata

from decimal import Decimal
from pathlib import Path

import pandas as pd
import torch
import bitsandbytes as bnb
from PIL import Image

from transformers import AutoProcessor, BitsAndBytesConfig, GenerationConfig, Qwen3VLForConditionalGeneration, Trainer, TrainerCallback, TrainingArguments, set_seed
from transformers.trainer_utils import get_last_checkpoint

from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training

In [15]:
ROOT = Path("/content/drive/MyDrive/MLLM/mordern_llm")
RUN_DIR = ROOT / "outputs/qlora_run1"
BEST_DIR = RUN_DIR / "best_adapter"
RUN_DIR.mkdir(parents=True, exist_ok=True)

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(value, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

base_cfg = read_json(
    ROOT / "outputs/quantization/qlora_base_config.json"
)
task_config = read_json(ROOT / "data/sroie/task_config.json")
manifest = read_json(ROOT / "data/sroie/manifest.json")

DOC_PROMPT = task_config["prompt"]
IMAGE_KWARGS = task_config["image_kwargs"]
MAX_SEQUENCE = task_config["max_sequence"]
FIELDS = tuple(task_config["fields"])

train_records = manifest["train"]
val_by_id = {r["id"]: r for r in manifest["val"]}
probe_records = [
    val_by_id[sample_id]
    for sample_id in manifest["val_probe_ids"]
]

DEVICE = torch.device("cuda:0")
set_seed(42)

print("GPU:", torch.cuda.get_device_name(0))
print("Training receipts:", len(train_records))
print("Validation probe:", len(probe_records))

GPU: Tesla T4
Training receipts: 551
Validation probe: 20


In [16]:
def load_base(for_training):
    quant_config = BitsAndBytesConfig.from_dict(
        base_cfg["quantization_config"]
    )
    net = Qwen3VLForConditionalGeneration.from_pretrained(
        base_cfg["model_id"],
        revision=base_cfg["revision"],
        quantization_config=quant_config,
        dtype=torch.float16,
        device_map={"": 0},
        attn_implementation="sdpa",
        low_cpu_mem_usage=True
    )
    
    net = prepare_model_for_kbit_training(
        net,
        use_gradient_checkpointing=for_training,
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )
    
    net.model.visual.to(dtype=torch.float16)
    net.config.use_cache = not for_training
    net.config.text_config.use_cache = not for_training
    return net

**LoRA adapters**

In [17]:
processor = AutoProcessor.from_pretrained(
    base_cfg["model_id"],
    revision=base_cfg["revision"],
    use_fast=False
)
base = load_base(for_training=True)

target_modules = [
    name for name, module in base.named_modules()
    if isinstance(module, bnb.nn.Linear4bit)
    and name.startswith("model.language_model.")
]

LORA_SETTINGS = {
    "r" : 8,
    "lora_alpha" : 16,
    "lora_dropout" : 0.05,
    "bias" : "none",
    "task_type": "CAUSAL_LM"
}

model = get_peft_model(
    base,
    LoraConfig(**LORA_SETTINGS, target_modules=target_modules),
    autocast_adapter_dtype=True
)

trainable = [
    (name, parameter) for name, parameter in model.named_parameters()
    if parameter.requires_grad
]

SMOKE_PASSED = False

print("Targeted linear layers:", len(target_modules))
model.print_trainable_parameters()
print("Example trainable parameter:", trainable[0][0])

/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Targeted linear layers: 196
trainable params: 8,716,288 || all params: 2,136,248,320 || trainable%: 0.4080
Example trainable parameter: base_model.model.model.language_model.layers.0.self_attn.q_proj.lora_A.default.weight


In [18]:
def training_messages(record):
    return user_messages() + [{
        "role": "assistant",
        "content": json.dumps(record["reference"], ensure_ascii=False)
    }]

def collate_receipts(records):
    processor.tokenizer.padding_side = "right"
    images = []
    full_texts = []
    prefix_texts = []
    
    for record in records:
        with Image.open(record["image_path"]) as source:
            images.append(source.convert("RGB"))
        
        full_texts.append(processor.apply_chat_template(
            training_messages(record),
            tokenize=False,
            add_generation_prompt=False
        ))
        prefix_texts.append(processor.apply_chat_template(
            user_messages(),
            tokenize=False,
            add_generation_prompt=True
        ))
    
    batch = processor(
        text=full_texts,
        images=images,
        images_kwargs=IMAGE_KWARGS,
        add_special_tokens=False,
        padding=True,
        return_tensors="pt"
    )
    prefix = processor(
        text=prefix_texts,
        images=images,
        images_kwargs=IMAGE_KWARGS,
        add_special_tokens=False,
        padding=True,
        return_tensors="pt"
    )
    
    labels = batch["input_ids"].clone()
    for index in range(len(records)):
        prefix_length = int(prefix["attention_mask"][index].sum())
        labels[index, :prefix_length] = -100
        
        
    labels[batch["attention_mask"] == 0] = -100
    labels[batch["input_ids"] == processor.image_token_id] = -100
    batch["labels"] = labels
    
    return batch

In [19]:
guard_record = max(
    train_records,
    key=lambda r: Path(r["image_path"]).stat().st_size
)

batch = collate_receipts([guard_record]).to(DEVICE)

model.train()
model.zero_grad(set_to_none=True)
torch.cuda.reset_peak_memory_stats()

with torch.autocast("cuda", dtype=torch.float16):
    loss = model(**batch).loss

loss.backward()

grads = [p.grad for _, p in trainable if p.grad is not None]
nonzero_grads = sum(g.abs().max().item() > 0 for g in grads)

print("Loss:", round(loss.item(), 4))
print("Nonzero adapter gradient tensors:", nonzero_grads)
print(
    "Forward/backward peak GiB:",
    round(torch.cuda.max_memory_allocated() / 2**30, 3),
)

model.zero_grad(set_to_none=True)
del batch, loss, grads

gc.collect()
torch.cuda.empty_cache()

SMOKE_PASSED = True
set_seed(42)

Loss: nan
Nonzero adapter gradient tensors: 0
Forward/backward peak GiB: 4.898


**Validation generation**

In [20]:
@torch.no_grad()
def generate_doc(net, proc, image_path):
    messages = [{
        "role" : "user",
        "content" : [
            {"type": "image"},
            {"type": "text", "text": DOC_PROMPT}
        ]
    }]
    
    text = proc.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    with Image.open(image_path) as source:
        image = source.convert("RGB")
    
    inputs = proc(
        text=[text],
        images=[image],
        images_kwargs=IMAGE_KWARGS,
        add_special_tokens=False,
        return_tensors="pt"
    ).to(DEVICE)
    
    generation = GenerationConfig(
        max_new_tokens=task_config["max_new_tokens"],
        do_sample=False,
        use_cache=True,
        eos_token_id=proc.tokenizer.eos_token_id,
        pad_token_id=proc.tokenizer.pad_token_id,
        bos_token_id=proc.tokenizer.bos_token_id
    )
    
    with torch.autocast("cuda", dtype=torch.float16):
        output = net.generate(
            **inputs,
            generation_config=generation,
            use_model_defaults=False
        )
    
    new_ids = output[:, inputs["input_ids"].shape[1]:]

    return proc.batch_decode(
        new_ids,
        skip_special_tokens=True
    )[0].strip()

In [21]:
METRIC_KEYS = [
    "raw_json_valid",
    "schema_valid",
    "company_correct",
    "date_correct",
    "total_correct",
    "all_correct"
]

class SelectAdapter(TrainerCallback):
    def __init__(self):
        history_path = RUN_DIR / "validation_history.json"
        best_path = BEST_DIR / "best_metrics.json"
        
        self.history = read_json(history_path) if history_path.exists() else []
        self.best_score = (
            tuple(read_json(best_path)["score"])
            if best_path.exists() else None
        )
    
    def on_epoch_end(
      self, args, state, control, model=None, **kwargs  
    ):
        was_training = model.training
        model.eval()
        rows = []
        
        try:
            for record in probe_records:
                answer = generate_doc(
                    model, processor, record["image_path"]
                )
                rows.append({
                    "id" : record["id"],
                    "answer": answer,
                    **score_receipt(answer, record["reference"])
                })
        finally:
            model.train(was_training)
        
        counts = {
            key: sum(row[key] for row in rows)
            for key in METRIC_KEYS
        }
        score = (
            counts["all_correct"],
            sum(counts[f"{field}_correct"] for field in FIELDS),
            counts["raw_json_valid"]
        )
        result = {
            "epoch": round(state.epoch, 3),
            "score": list(score),
            **{
                f"{key}_pct": 100 * count / len(rows)
                for key, count in counts.items()
            },
        }

        self.history.append(result)
        save_json(RUN_DIR / "validation_history.json", self.history)

        if self.best_score is None or score > self.best_score:
            self.best_score = score
            BEST_DIR.mkdir(parents=True, exist_ok=True)

            model.save_pretrained(BEST_DIR, safe_serialization=True,)
            processor.save_pretrained(BEST_DIR)

            save_json(BEST_DIR / "best_metrics.json", result)
            save_json(BEST_DIR / "validation_rows.json", rows)
            save_json(BEST_DIR / "task_config.json", task_config)
            save_json(BEST_DIR / "training_recipe.json", recipe)

        print("\nValidation:", result)
        control.should_save = True
        return control

**training**